# Hari 001c — Opsi Asia: Monte Carlo, variance reduction & Greeks

| Hari | Tanggal | Jenis | Topik |
|------|---------|-------|-------|
| 001c | 2026-11-19 | `quant` | Opsi Asia: Monte Carlo, variance reduction & Greeks |

> Bagian dari **100QB** — 100 Days of Quant + Kebun Simulasi.

## Konsep

### 1. Opsi Asia Aritmetik (*Arithmetic Asian Call*)
Opsi Asia bergantung pada rata-rata harga aset sepanjang periode pemantauan diskret $t_1, \dots, t_M$ ($M = 252$, $\Delta t = T/M$):

$$A = \frac{1}{M} \sum_{i=1}^{M} S_{t_i}, \qquad \text{Payoff} = \max(A - K, 0)$$

Di bawah model *Geometric Brownian Motion* (GBM) netral-risiko, harga aset berevolusi sebagai $S_{t_i} = S_{t_{i-1}} \exp\!\left((r - \tfrac{1}{2}\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\,Z_i\right)$. Berbeda dengan rata-rata geometrik, distribusi jumlah variabel lognormal ($A$) tidak memiliki rumus tertutup sederhana sehingga memerlukan simulasi Monte Carlo.

### 2. *Variance Reduction*: Antithetic & Control Variate
- **Antithetic Variates**: Memasangkan setiap jalur acak $Z$ dengan $-Z$ untuk menciptakan korelasi negatif antar pasangan sampel sehingga varians rata-rata pasangan turun.
- **Control Variate (Opsi Asia Geometrik)**: Memakai payoff terdiskonto opsi Asia geometrik $Y = e^{-rT}\max(G - K, 0)$ dengan $G = (\prod_{i=1}^M S_{t_i})^{1/M}$, yang memiliki nilai ekspektasi tertutup diskret $\mathbb{E}[Y]$ (Kemna–Vorst). Estimator terkontrol:

$$\hat{C}_{\text{CV}} = \bar{X} - \hat{\beta}(\bar{Y} - \mathbb{E}[Y]), \qquad \hat{\beta} = \frac{\widehat{\text{Cov}}(X, Y)}{\widehat{\text{Var}}(Y)}$$

Karena $A$ dan $G$ berkorelasi sangat tinggi ($> 0.99$), varians $X - \hat{\beta} Y$ turun drastis.

### 3. Turunan Singkat Delta *Pathwise*
Karena $S_{t_i}(S_0) = S_0 \cdot \exp(\dots)$ proporsional terhadap $S_0$, maka $\frac{\partial A}{\partial S_0} = \frac{A}{S_0}$. Dengan menukar turunan dan ekspektasi (valid karena payoff kontinu dan diferensiabel hampir di mana-mana):

$$\Delta_{\text{PW}} = \frac{\partial}{\partial S_0} e^{-rT}\mathbb{E}[\max(A - K, 0)] = e^{-rT}\mathbb{E}\!\left[\mathbf{1}_{\{A > K\}} \frac{A}{S_0}\right]$$

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "qb100").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qb100 import options as opt, plotting

plotting.set_style()

S0 = 100.0
K = 100.0
r = 0.05
sigma = 0.20
T = 1.0
N_STEPS = 252
N_PATHS = 200_000
SEED = 42

## Implementasi

In [2]:
# 1. Simulasi 200.000 jalur GBM + Sanity Check Call Eropa (BS vs MC)
paths = opt.simulate_gbm_paths(S0, r, sigma, T, N_STEPS, N_PATHS, seed=SEED)
bs_euro = opt.bs_call(S0, K, r, sigma, T)
mc_euro = opt.price_european_mc(paths, K, r, T)
diff_euro = mc_euro['price'] - bs_euro
z_euro = abs(diff_euro) / mc_euro['se']

print(f"Black-Scholes Eropa : {bs_euro:.6f}")
print(f"Monte Carlo Eropa   : {mc_euro['price']:.6f} ± {mc_euro['se']:.6f}")
print(f"Selisih             : {diff_euro:+.6f} ({z_euro:.2f} SE; harus < 3 SE)")
assert z_euro < 3.0

Black-Scholes Eropa : 10.450584
Monte Carlo Eropa   : 10.504703 ± 0.033110
Selisih             : +0.054119 (1.63 SE; harus < 3 SE)


In [3]:
# 2. Tiga estimator harga Arithmetic Asian Call (200.000 path)
res_naive = opt.price_asian_naive(S0, K, r, sigma, T, N_STEPS, N_PATHS, paths=paths)
res_anti = opt.price_asian_antithetic(S0, K, r, sigma, T, N_STEPS, N_PATHS, seed=SEED)
res_cv = opt.price_asian_control_variate(S0, K, r, sigma, T, N_STEPS, N_PATHS, paths=paths)

var_naive = res_naive['se'] ** 2
est_rows = []
for name, res in [
    ("Naive MC (200k)", res_naive),
    ("Antithetic (100k pasang)", res_anti),
    ("Control Variate (Geom. Asia)", res_cv),
]:
    vrf = var_naive / (res['se'] ** 2)
    est_rows.append({
        "Estimator": name,
        "Harga": res['price'],
        "SE": res['se'],
        "CI 95% Bawah": res['ci_low'],
        "CI 95% Atas": res['ci_high'],
        "VRF (Var_naive / Var)": vrf,
    })

est_df = pd.DataFrame(est_rows)
print(f"Harga tertutup Geometric Asian : {res_cv['y_exact']:.6f}")
print(f"Koefisien optimal beta         : {res_cv['beta']:.6f}")
est_df

Harga tertutup Geometric Asian : 5.565509
Koefisien optimal beta         : 1.035673


,Estimator,Harga,SE,CI 95% Bawah,CI 95% Atas,VRF (Var_naive / Var)
0,Naive MC (200k),5.804757,0.017935,5.769605,5.839910,1.000000
1,Antithetic (100k pasang),5.791835,0.012451,5.767432,5.816239,2.074945
2,Control Variate (Geom. Asia),5.782379,0.000498,5.781402,5.783355,1295.285221


In [4]:
# 3. Delta Opsi Asia: Pathwise vs Bump-and-Revalue (CRN, h = 1% * S0)
delta_pw = opt.delta_asian_pathwise(S0, K, r, sigma, T, N_STEPS, N_PATHS, paths=paths)
delta_bmp = opt.delta_asian_bump(S0, K, r, sigma, T, N_STEPS, N_PATHS, h_rel=0.01, paths=paths)
delta_diff = delta_pw['delta'] - delta_bmp['delta']
delta_se_comb = math.hypot(delta_pw['se'], delta_bmp['se'])
delta_z = abs(delta_diff) / delta_se_comb

delta_df = pd.DataFrame([
    {"Metode": "Pathwise", "Delta": delta_pw['delta'], "SE": delta_pw['se'], "CI 95%": f"[{delta_pw['ci_low']:.5f}, {delta_pw['ci_high']:.5f}]"},
    {"Metode": "Bump-and-Revalue (h=1%)", "Delta": delta_bmp['delta'], "SE": delta_bmp['se'], "CI 95%": f"[{delta_bmp['ci_low']:.5f}, {delta_bmp['ci_high']:.5f}]"},
])
print(f"Selisih Delta (PW - Bump): {delta_diff:+.6f} ({delta_z:.2f} SE gabungan; konsisten < 3 SE)")
assert delta_z < 3.0
delta_df

Selisih Delta (PW - Bump): +0.000040 (0.02 SE gabungan; konsisten < 3 SE)


,Metode,Delta,SE,CI 95%
0,Pathwise,0.589628,0.001179,"[0.58732, 0.59194]"
1,Bump-and-Revalue (h=1%),0.589588,0.001158,"[0.58732, 0.59186]"


In [5]:
# 4. Analisis Konvergensi SE vs N untuk ketiga estimator
n_grid = [1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000]
conv_rows = []
for n in n_grid:
    p_sub = paths[:n]
    se_n = opt.price_asian_naive(S0, K, r, sigma, T, N_STEPS, n, paths=p_sub)['se']
    se_a = opt.price_asian_antithetic(S0, K, r, sigma, T, N_STEPS, n, seed=SEED)['se']
    se_c = opt.price_asian_control_variate(S0, K, r, sigma, T, N_STEPS, n, paths=p_sub)['se']
    conv_rows.append({"N": n, "SE_Naive": se_n, "SE_Antithetic": se_a, "SE_ControlVariate": se_c})

conv_df = pd.DataFrame(conv_rows)
slopes = {
    col: float(np.polyfit(np.log10(conv_df["N"]), np.log10(conv_df[col]), 1)[0])
    for col in ["SE_Naive", "SE_Antithetic", "SE_ControlVariate"]
}
print("Slope log-log (teori = -0.50):")
for k, v in slopes.items():
    print(f"  {k:18s}: {v:.4f}")
conv_df

Slope log-log (teori = -0.50):
  SE_Naive          : -0.5087
  SE_Antithetic     : -0.5022
  SE_ControlVariate : -0.4991


,N,SE_Naive,SE_Antithetic,SE_ControlVariate
0,1000,0.268002,0.178696,0.007086
1,2000,0.184652,0.125842,0.004863
2,5000,0.114097,0.078532,0.003197
3,10000,0.080748,0.054933,0.002324
4,20000,0.057062,0.039307,0.001617
5,50000,0.035753,0.024971,0.001001
6,100000,0.025408,0.017571,0.000707
7,200000,0.017935,0.012451,0.000498


## Visualisasi

In [6]:
# Grafik 1: Konvergensi SE vs N (skala log-log) + slope terukur
fig, ax = plt.subplots()
for col, label in [
    ("SE_Naive", f"Naive MC (slope={slopes['SE_Naive']:.3f})"),
    ("SE_Antithetic", f"Antithetic (slope={slopes['SE_Antithetic']:.3f})"),
    ("SE_ControlVariate", f"Control Variate (slope={slopes['SE_ControlVariate']:.3f})"),
]:
    ax.loglog(conv_df["N"], conv_df[col], "o-", linewidth=1.8, label=label)

ax.set_title("Konvergensi Standard Error vs Jumlah Path (Log-Log)")
ax.set_xlabel("Jumlah Path $N$")
ax.set_ylabel("Standard Error (SE)")
ax.legend()
plotting.save_figure(fig, "day001c_asian-convergence-loglog")
plt.show()

/var/folders/ms/5n6sfq8j2z52l34_szzbz02w0000gn/T/ipykernel_56923/3966802315.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Grafik 2: Perbandingan Estimasi Harga (CI 95%) dan Variance Reduction Factor
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

y_pos = np.arange(len(est_df))
ax1.errorbar(
    est_df["Harga"], y_pos,
    xerr=1.96 * est_df["SE"],
    fmt="o", capsize=5, linewidth=2,
)
ax1.axvline(res_cv["price"], color="gray", linestyle="--", linewidth=1)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(est_df["Estimator"])
ax1.set_title("Estimasi Harga + CI 95% (200k Path)")
ax1.set_xlabel("Harga Opsi Asia")

ax2.barh(y_pos, est_df["VRF (Var_naive / Var)"], color=plotting.PALETTE[:3])
ax2.set_xscale("log")
ax2.set_yticks(y_pos)
ax2.set_yticklabels(["Naive", "Antithetic", "Control Variate"])
ax2.set_title("Variance Reduction Factor (Skala Log)")
ax2.set_xlabel("Var(Naive) / Var(Metode)")
for idx, val in enumerate(est_df["VRF (Var_naive / Var)"]):
    ax2.text(val * 1.1, idx, f"{val:,.1f}x", va="center", fontsize=9)

fig.tight_layout()
plotting.save_figure(fig, "day001c_asian-estimators-ci")
plt.show()

/var/folders/ms/5n6sfq8j2z52l34_szzbz02w0000gn/T/ipykernel_56923/3576039183.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Kesimpulan (3 poin)

1. **Control variate** dengan opsi Asia geometrik diskret (Kemna–Vorst) memangkas varians secara drastis dibanding *naive* MC dan *antithetic variates*, sehingga menghasilkan CI 95% paling sempit pada jumlah path yang sama.
2. Ketiga estimator mengikuti laju konvergensi teoretis Monte Carlo $\text{SE} \propto N^{-0.5}$ (slope log-log $\approx -0.50$); peningkatan efisiensi murni berasal dari konstanta varians yang jauh lebih kecil.
3. Estimasi **Delta** dengan metode *pathwise* dan *bump-and-revalue* (CRN, $h = 1\%$) konsisten dalam $< 3\text{ SE}$, namun *pathwise* tidak memerlukan simulasi ulang harga terganggu sehingga lebih efisien.

## Referensi

- Kemna, A. G. Z., & Vorst, A. C. F. (1990). *A Pricing Method for Options Based on Average Asset Values*. Journal of Banking & Finance.
- Glasserman, P. (2003). *Monte Carlo Methods in Financial Engineering* (Bab 4: Variance Reduction, Bab 7: Sensitivities). Springer.
- Hull, J. C. *Options, Futures, and Other Derivatives* — Exotic Options (Asian Options).